In [8]:
from dotenv import load_dotenv
import os
import pandas as pd
import requests
from getpass import getpass

In [ ]:

load_dotenv()  # carrega variáveis de ambiente do arquivo .env, se existir
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")
if not GOOGLE_MAPS_API_KEY:
    raise RuntimeError("Defina GOOGLE_MAPS_API_KEY no arquivo .env (veja .env.example) ou como variável de ambiente.")

In [ ]:
API_KEY = getpass("Cole sua Google Maps API key: ")

In [21]:
def get_lat_lng(cidade: str, api_key: str, pais: str = "BR") -> dict:
    """
    Recebe o nome de uma cidade e retorna um dicionário com lat, lng
    e o endereço formatado retornado pelo Google.

    pais: código ISO 3166-1 Alpha-2 do país para RESTRINGIR a busca
          (ex: "BR" para Brasil). Passe None para buscar sem restrição
          de país.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": cidade, "key": api_key}

    if pais:
        params["components"] = f"country:{pais}"

    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    status = data.get("status")
    if status != "OK":
        erro = data.get("error_message", "")
        raise ValueError(f"Geocoding falhou (status={status}). {erro}")

    resultado = data["results"][0]
    location = resultado["geometry"]["location"]

    return {
        "cidade_buscada": cidade,
        "endereco_formatado_cidade_datacenter": resultado.get("formatted_address"),
        "latitude_cidade_datacenter": location["lat"],
        "longitude_cidade_datacenter": location["lng"],
    }

In [22]:
info = get_lat_lng('Barueri', GOOGLE_MAPS_API_KEY)

In [23]:
info

{'cidade_buscada': 'Barueri',
 'endereco_formatado_cidade_datacenter': 'Barueri - São Paulo, Brazil',
 'latitude_cidade_datacenter': -23.5035038,
 'longitude_cidade_datacenter': -46.8785555}

In [17]:
dados_datacenters = pd.read_csv("../facilities_brazil_peering.csv")

In [15]:
dados_datacenters['city']

0           Barueri
1         São Paulo
2    Rio de Janeiro
3             Cotia
4         São Paulo
Name: city, dtype: str

In [16]:
dados_datacenters

,id,org_id,org_name,campus_id,name,aka,name_long,website,social_media,clli,...,address1,address2,city,country,state,zipcode,floor,suite,latitude,longitude
0,165,2,"Equinix, Inc.",NaN,Equinix SP4 - São Paulo,NaN,NaN,https://www.equinix.com/locations/americas-col...,"{'service': 'website', 'identifier': 'https://...",NaN,...,"Av. Ceci, 1900 - Res. Tambore",Tamboré,Barueri,BR,SP,06455000,NaN,NaN,-23.497589,-46.814577
1,816,20869,Telium Telecomunicações Ltda,NaN,PIX Telium,Telium,Telium Telecomunicações Ltda,http://www.telium.com.br,"{'service': 'website', 'identifier': 'http://w...",NaN,...,Av. Nações unidas 13797 Building III 2° floor,NaN,São Paulo,BR,SP,04794-000,NaN,NaN,-23.620044,-46.702253
2,1025,3820,Moebius Tecnologia,NaN,Moebius Datacenter,NaN,NaN,http://www.moebius.com.br,"{'service': 'website', 'identifier': 'http://w...",NaN,...,Rua Jardim Botanico 674 sala 507,NaN,Rio de Janeiro,BR,RJ,22461000,NaN,NaN,-22.964262,-43.218326
3,1057,35201,Cirion,NaN,Cirion Sao Paulo - SAO1,Teleporto Cotia,Cirion Technologies,https://ciriontechnologies.com,"{'service': 'website', 'identifier': 'https://...",NaN,...,"Av. Eid Mansur, 666",Parque Sao George,Cotia,BR,NaN,06708-070,NaN,NaN,-23.597933,-46.848808
4,1099,4855,G8,NaN,PIX Samm Florida,NaN,NaN,http://www.g8.net.br,"{'service': 'website', 'identifier': 'http://w...",NaN,...,"R. Flórida, 1738 – Conj. Comercial 31, 3º anda...",NaN,São Paulo,BR,SP,04575901,NaN,NaN,-23.607007,-46.694739


In [24]:
import time

# cache para nao repetir chamada pra API quando a mesma cidade aparecer em varias linhas
cache_geocoding = {}

def get_info_cidade(cidade):
    if cidade in cache_geocoding:
        return cache_geocoding[cidade]

    try:
        info = get_lat_lng(cidade, GOOGLE_MAPS_API_KEY)
    except Exception as e:
        print(f"Erro ao buscar '{cidade}': {e}")
        info = {
            "endereco_formatado_cidade_datacenter": None,
            "latitude_cidade_datacenter": None,
            "longitude_cidade_datacenter": None,
        }

    cache_geocoding[cidade] = info
    return info


enderecos = []
latitudes = []
longitudes = []

for cidade in dados_datacenters['city']:
    info = get_info_cidade(cidade)
    enderecos.append(info.get('endereco_formatado_cidade_datacenter'))
    latitudes.append(info.get('latitude_cidade_datacenter'))
    longitudes.append(info.get('longitude_cidade_datacenter'))
    time.sleep(0.05)  # opcional, evita bater rate limit se a lista for grande

dados_datacenters['endereco_formatado_cidade_datacenter'] = enderecos
dados_datacenters['latitude_cidade_datacenter'] = latitudes
dados_datacenters['longitude_cidade_datacenter'] = longitudes

In [26]:
dados_datacenters.to_csv("../facilities_brazil_peering_com_lat_lng.csv", index=False)